# Airbnb Price Prediction

**Task:** Predict `log_price` — the natural log of an Airbnb listing's nightly price — from listing characteristics.

This is a regression problem. The dataset covers ~22k listings across 6 US cities (NYC, LA, SF, DC, Chicago, Boston). About half the columns are clean numerics; the rest are text, dates, free-form strings, and a particularly messy `amenities` field that needs parsing.

One thing worth noting upfront: `description` and `name` are truncated at 1000 characters in this dataset, so sophisticated NLP won't extract much beyond basic length and keyword features. The real action is in structured features — room type, accommodates, city — which explain most of the variance.

**Target:** `log_price` ∈ [2.3, 7.6], mean ≈ 4.78 (~$119/night in raw price).

Submission format: `Unnamed: 0` (listing id) + `logpred`.

## 1. Imports and data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import KFold, cross_val_score, cross_validate
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

try:
    import xgboost as xgb
    import lightgbm as lgb
    BOOSTING = True
    print("XGBoost and LightGBM available.")
except ImportError:
    BOOSTING = False
    print("xgboost/lightgbm not installed — using sklearn GBM. Install with: pip install xgboost lightgbm")

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
np.random.seed(42)

In [ ]:
DATA_PATH = ""  # adjust if CSVs are in a subfolder

train  = pd.read_csv(DATA_PATH + "airbnb_train.csv")
test   = pd.read_csv(DATA_PATH + "airbnb_test.csv")
sample = pd.read_csv(DATA_PATH + "prediction_example.csv")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Sample: {sample.shape}")
print()
print("Target (log_price) stats:")
print(train['log_price'].describe())

## 2. Dataset inspection

In [ ]:
train.head(3)

In [ ]:
print("Column types:")
print(train.dtypes)
print()
print("Missing values:")
miss = train.isnull().sum()
print(miss[miss > 0].sort_values(ascending=False))

In [ ]:
# Missing values heatmap
fig, ax = plt.subplots(figsize=(10, 4))
miss_pct = train.isnull().mean().sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]
ax.barh(miss_pct.index, miss_pct.values * 100, color='#E57373')
ax.set_xlabel('Missing %')
ax.set_title('Missing value rate per column')
ax.axvline(10, color='gray', linestyle='--', linewidth=0.8, label='10% threshold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig_missing.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(train['log_price'], bins=60, color='#42A5F5', edgecolor='white', linewidth=0.3)
axes[0].set_title('log_price distribution (train)', fontsize=12)
axes[0].set_xlabel('log_price')
axes[0].set_ylabel('Count')
axes[0].axvline(train['log_price'].mean(), color='red', linestyle='--', linewidth=1.2, label=f"Mean = {train['log_price'].mean():.2f}")
axes[0].legend(fontsize=9)

# Raw price distribution (exp of log_price)
raw_price = np.exp(train['log_price'])
axes[1].hist(raw_price[raw_price < 1000], bins=60, color='#66BB6A', edgecolor='white', linewidth=0.3)
axes[1].set_title('Raw price distribution (capped at $1000)', fontsize=12)
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('fig_target_dist.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Price range: ${np.exp(train['log_price'].min()):.0f} – ${np.exp(train['log_price'].max()):.0f}")
print(f"Median price: ${np.exp(train['log_price'].median()):.0f}/night")

In [ ]:
# Price by city
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

city_price = train.groupby('city')['log_price'].mean().sort_values(ascending=False)
axes[0].bar(city_price.index, np.exp(city_price.values),
            color=['#F44336','#E91E63','#9C27B0','#3F51B5','#2196F3','#009688'])
axes[0].set_title('Median nightly price by city', fontsize=12)
axes[0].set_ylabel('Price ($)')

# Violin plot
city_order = train.groupby('city')['log_price'].median().sort_values(ascending=False).index.tolist()
sns.violinplot(data=train, x='city', y='log_price', order=city_order,
               palette='Set2', ax=axes[1], inner='quartile')
axes[1].set_title('log_price distribution by city', fontsize=12)
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('fig_price_city.png', dpi=120, bbox_inches='tight')
plt.show()

print(train.groupby('city')['log_price'].agg(['mean','std','count']).round(3).to_string())

In [ ]:
# Price by room type and property type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rt_order = train.groupby('room_type')['log_price'].median().sort_values(ascending=False).index
sns.boxplot(data=train, x='room_type', y='log_price', order=rt_order,
            palette='Blues_r', ax=axes[0])
axes[0].set_title('log_price by room type')
axes[0].set_xlabel('')

# Top 8 property types
top_pt = train['property_type'].value_counts().head(8).index
pt_data = train[train['property_type'].isin(top_pt)]
pt_order = pt_data.groupby('property_type')['log_price'].median().sort_values(ascending=False).index
sns.boxplot(data=pt_data, x='property_type', y='log_price', order=pt_order,
            palette='Greens_r', ax=axes[1])
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_title('log_price by property type (top 8)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('fig_price_roomtype.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Price vs accommodates — clearest continuous relationship
fig, ax = plt.subplots(figsize=(10, 4))
acc_price = train[train['accommodates'] <= 12].groupby('accommodates')['log_price'].mean()
ax.plot(acc_price.index, acc_price.values, marker='o', linewidth=2, color='#5C6BC0')
ax.fill_between(acc_price.index,
                train[train['accommodates'] <= 12].groupby('accommodates')['log_price'].quantile(0.25),
                train[train['accommodates'] <= 12].groupby('accommodates')['log_price'].quantile(0.75),
                alpha=0.2, color='#5C6BC0')
ax.set_title('Mean log_price by accommodates (shading = IQR)')
ax.set_xlabel('Accommodates (guests)')
ax.set_ylabel('Mean log_price')
plt.tight_layout()
plt.savefig('fig_price_accommodates.png', dpi=120, bbox_inches='tight')
plt.show()

# Correlation of numeric columns with target
num_cols = ['accommodates', 'bathrooms', 'bedrooms', 'beds', 'number_of_reviews', 'review_scores_rating']
corr_with_price = train[num_cols + ['log_price']].corr()['log_price'].drop('log_price').sort_values(key=abs, ascending=False)
print("Numeric feature correlations with log_price:")
print(corr_with_price.round(3).to_string())

## 3. Geographic analysis

The latitude/longitude data is worth exploring — prices vary significantly within cities, not just between them.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
cities = ['NYC', 'LA', 'SF', 'DC', 'Chicago', 'Boston']
axes = axes.flatten()

for i, city in enumerate(cities):
    sub = train[train['city'] == city]
    sc = axes[i].scatter(sub['longitude'], sub['latitude'],
                         c=sub['log_price'], cmap='YlOrRd',
                         alpha=0.4, s=8, vmin=3.5, vmax=6.5)
    axes[i].set_title(f'{city} (n={len(sub):,})', fontsize=11)
    axes[i].set_xlabel('Longitude')
    axes[i].set_ylabel('Latitude')
    plt.colorbar(sc, ax=axes[i], label='log_price')

plt.suptitle('Airbnb price heatmap by city', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('fig_geo.png', dpi=120, bbox_inches='tight')
plt.show()
print("Clear geographic price clustering within each city — lat/lon are useful features.")

## 4. Feature engineering

The biggest opportunities here are:
1. **Parsing `amenities`** — it's a set-formatted string; count of amenities + flags for specific ones
2. **Parsing dates** — host experience, review recency
3. **Text features** — description length, keyword presence
4. **Cleaning binary columns** — t/f strings to 0/1
5. **Encoding high-cardinality columns** — neighbourhood, zipcode

Everything gets applied to both train and test identically.

In [ ]:
# ── 4.1 Combine train and test for consistent preprocessing ──
# Easier to engineer features on the combined set

train['_split'] = 'train'
test['_split'] = 'test'

# Align columns — test has 'Unnamed: 0' as id, train has 'id' + 'log_price'
test = test.rename(columns={'Unnamed: 0': 'id'})
# test doesn't have log_price — add placeholder
test['log_price'] = np.nan

df = pd.concat([train, test], ignore_index=True, sort=False)
print(f"Combined dataframe: {df.shape}")
print(f"Train rows: {(df['_split']=='train').sum()}, Test rows: {(df['_split']=='test').sum()}")

In [ ]:
# ── 4.2 Boolean columns: t/f → 1/0 ──
bool_cols = ['host_has_profile_pic', 'host_identity_verified', 'instant_bookable']
for col in bool_cols:
    df[col] = df[col].map({'t': 1, 'f': 0, True: 1, False: 0})
    df[col] = pd.to_numeric(df[col], errors='coerce')

# cleaning_fee is already bool in train but might be string in test
df['cleaning_fee'] = df['cleaning_fee'].map({True: 1, False: 0, 't': 1, 'f': 0}).fillna(0).astype(int)

print("Boolean columns converted:")
print(df[bool_cols + ['cleaning_fee']].head(5).to_string())

In [ ]:
# ── 4.3 Host response rate: '100%' → 1.0 ──
df['host_response_rate_num'] = (
    df['host_response_rate']
    .str.replace('%', '', regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors='coerce')
    / 100
)
print(f"Host response rate: {df['host_response_rate_num'].describe()}")

In [ ]:
# ── 4.4 Date features ──
for col in ['host_since', 'first_review', 'last_review']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

reference_date = pd.Timestamp('2017-12-31')  # approx dataset cutoff

# Host experience in years
df['host_experience_years'] = (reference_date - df['host_since']).dt.days / 365.25

# Days since first review (a proxy for listing maturity)
df['days_since_first_review'] = (reference_date - df['first_review']).dt.days

# Days since last review (recency — stale listings might be inactive)
df['days_since_last_review'] = (reference_date - df['last_review']).dt.days

# Review span: how many days between first and last review
df['review_span_days'] = (df['last_review'] - df['first_review']).dt.days

# Flag: has any reviews
df['has_reviews'] = (~df['first_review'].isna()).astype(int)

print("Date features sample:")
print(df[['host_experience_years','days_since_first_review','days_since_last_review','review_span_days']].head(5).round(1).to_string())

In [ ]:
# ── 4.5 Amenities parsing ──
# Format: {TV,"Wireless Internet",Kitchen,...}
# Extract: count + specific valuable amenity flags

def parse_amenities(s):
    if pd.isna(s):
        return set()
    # Remove braces and split by comma, strip quotes
    s = s.strip('{}')
    items = re.split(r',(?=(?:[^"]*"[^"]*")*[^"]*$)', s)
    return {item.strip().strip('"').lower() for item in items}

df['amenities_set'] = df['amenities'].apply(parse_amenities)
df['amenities_count'] = df['amenities_set'].apply(len)

# High-value amenity flags (correlated with premium listings)
valuable_amenities = {
    'has_wifi':         ['wireless internet', 'wifi'],
    'has_tv':           ['tv'],
    'has_kitchen':      ['kitchen'],
    'has_parking':      ['free parking on premises', 'free parking'],
    'has_ac':           ['air conditioning'],
    'has_heating':      ['heating'],
    'has_washer':       ['washer'],
    'has_dryer':        ['dryer'],
    'has_doorman':      ['doorman'],
    'has_elevator':     ['elevator in building', 'elevator'],
    'has_pool':         ['pool'],
    'has_gym':          ['gym'],
    'has_hot_tub':      ['hot tub'],
    'has_pets_allowed': ['pets allowed'],
    'has_breakfast':    ['breakfast'],
    'has_workspace':    ['laptop friendly workspace'],
    'has_checkin_24h':  ['24-hour check-in'],
}

for feat, keywords in valuable_amenities.items():
    df[feat] = df['amenities_set'].apply(
        lambda s: int(any(k in s for k in keywords))
    )

print(f"Amenities features added: {len(valuable_amenities)} binary flags + count")
print(f"Average amenities count: {df['amenities_count'].mean():.1f}")
# Verify correlation with price in train
amenity_corrs = df[df['_split']=='train'][[k for k in valuable_amenities.keys()] + ['amenities_count','log_price']].corr()['log_price'].drop('log_price').sort_values(key=abs, ascending=False)
print("\nAmenity correlations with log_price:")
print(amenity_corrs.round(3).to_string())

In [ ]:
# ── 4.6 Text features from description and name ──

# Description
df['desc_length'] = df['description'].fillna('').apply(len)
df['desc_word_count'] = df['description'].fillna('').apply(lambda x: len(x.split()))

# Luxury keywords in description
luxury_keywords = ['luxury', 'premium', 'exclusive', 'stunning', 'penthouse',
                   'rooftop', 'concierge', 'gourmet', 'spacious', 'designer']
df['desc_luxury_score'] = df['description'].fillna('').str.lower().apply(
    lambda x: sum(1 for kw in luxury_keywords if kw in x)
)

# Name length (longer names sometimes reflect more descriptive/premium listings)
df['name_length'] = df['name'].fillna('').apply(len)

print("Text features:")
print(df[['desc_length','desc_word_count','desc_luxury_score','name_length']].describe().round(1).to_string())

In [ ]:
# ── 4.7 Neighbourhood encoding ──
# High cardinality column (>1000 unique values) — target encode in train,
# apply to test using train stats

# Compute target mean per neighbourhood on train only (to avoid leakage)
train_mask = df['_split'] == 'train'

neigh_target_mean = df[train_mask].groupby('neighbourhood')['log_price'].mean()
neigh_freq = df[train_mask]['neighbourhood'].value_counts()

# Apply: if neighbourhood seen in train, use target mean; else use global mean
global_mean = df[train_mask]['log_price'].mean()
df['neighbourhood_mean_price'] = df['neighbourhood'].map(neigh_target_mean).fillna(global_mean)

# Also frequency encode (useful for tree models)
df['neighbourhood_freq'] = df['neighbourhood'].map(neigh_freq).fillna(1)

print(f"Neighbourhood target encoding done. {df['neighbourhood'].nunique()} unique neighbourhoods.")
print(f"Top 5 by mean price:")
print(neigh_target_mean.sort_values(ascending=False).head(5).round(3).to_string())

In [ ]:
# ── 4.8 Zipcode cleaning ──
# Zipcodes are messy (NaN, wrong format, partial codes)
df['zipcode_clean'] = df['zipcode'].astype(str).str.strip()
df['zipcode_clean'] = df['zipcode_clean'].str.extract(r'(\d{5})', expand=False)  # keep first 5 digits

# Encode as frequency in train
zip_mean = df[train_mask].groupby('zipcode_clean')['log_price'].mean()
df['zipcode_mean_price'] = df['zipcode_clean'].map(zip_mean).fillna(global_mean)

print(f"Zipcode encoding done. {df['zipcode_clean'].nunique()} unique 5-digit zipcodes.")

In [ ]:
# ── 4.9 Categorical encoding ──
# Low-cardinality categoricals: label encode or one-hot
# room_type, bed_type, cancellation_policy, property_type, city

# Room type: ordinal makes sense (Entire > Private > Shared)
room_type_map = {'Entire home/apt': 2, 'Private room': 1, 'Shared room': 0}
df['room_type_ord'] = df['room_type'].map(room_type_map).fillna(1)

# Cancellation policy: ordinal (more flexible = lower barrier = maybe lower price?)
cancel_map = {'flexible': 0, 'moderate': 1, 'strict': 2, 'super_strict_30': 3, 'super_strict_60': 4}
df['cancellation_ord'] = df['cancellation_policy'].map(cancel_map).fillna(1)

# City: mean encode (city is a strong signal)
city_mean = df[train_mask].groupby('city')['log_price'].mean()
df['city_mean_price'] = df['city'].map(city_mean)

# Property type: group rare types and mean encode
df['property_type_clean'] = df['property_type'].copy()
top_prop_types = df[train_mask]['property_type'].value_counts().head(8).index
df.loc[~df['property_type_clean'].isin(top_prop_types), 'property_type_clean'] = 'Other'
prop_mean = df[train_mask].groupby('property_type_clean')['log_price'].mean()
df['property_type_mean_price'] = df['property_type_clean'].map(prop_mean).fillna(global_mean)

# Bed type: Real Bed is dominant — flag it
df['is_real_bed'] = (df['bed_type'] == 'Real Bed').astype(int)

print("Categorical encodings done.")
print("City mean prices:")
print(city_mean.sort_values(ascending=False).round(3).to_string())

In [ ]:
# ── 4.10 Interaction and derived features ──

# Price per person proxy: accommodates / bedrooms ratio
df['accommodates_per_bedroom'] = df['accommodates'] / (df['bedrooms'].fillna(1) + 0.5)

# Bed density: beds / accommodates (low ratio = crowded)
df['bed_per_person'] = df['beds'].fillna(1) / (df['accommodates'] + 0.01)

# Bathroom ratio
df['bath_per_bedroom'] = df['bathrooms'].fillna(1) / (df['bedrooms'].fillna(1) + 0.5)

# Log-transform skewed numerics (number_of_reviews has heavy right tail)
df['log_reviews'] = np.log1p(df['number_of_reviews'])
df['log_amenities_count'] = np.log1p(df['amenities_count'])

print("Derived features added.")
print(df[['accommodates_per_bedroom','bed_per_person','bath_per_bedroom','log_reviews']].describe().round(2).to_string())

In [ ]:
# ── Final feature set ──
FEATURES = [
    # Core numeric
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'number_of_reviews', 'review_scores_rating',
    # Engineered numeric
    'amenities_count', 'log_amenities_count', 'log_reviews',
    'desc_length', 'desc_word_count', 'desc_luxury_score', 'name_length',
    'host_experience_years', 'days_since_first_review',
    'days_since_last_review', 'review_span_days',
    'host_response_rate_num',
    'accommodates_per_bedroom', 'bed_per_person', 'bath_per_bedroom',
    # Boolean
    'cleaning_fee', 'host_has_profile_pic', 'host_identity_verified',
    'instant_bookable', 'has_reviews', 'is_real_bed',
    # Amenity flags
    'has_wifi', 'has_tv', 'has_kitchen', 'has_parking', 'has_ac',
    'has_heating', 'has_washer', 'has_dryer', 'has_doorman',
    'has_elevator', 'has_pool', 'has_gym', 'has_hot_tub',
    'has_pets_allowed', 'has_breakfast', 'has_workspace', 'has_checkin_24h',
    # Encoded categoricals
    'room_type_ord', 'cancellation_ord', 'city_mean_price',
    'property_type_mean_price', 'neighbourhood_mean_price',
    'neighbourhood_freq', 'zipcode_mean_price',
    # Geographic
    'latitude', 'longitude',
]

FEATURES = [f for f in FEATURES if f in df.columns]
print(f"Total features: {len(FEATURES)}")
print("Missing columns:", [f for f in FEATURES if f not in df.columns])

## 5. Feature analysis

In [ ]:
# Correlation with target
train_df = df[df['_split'] == 'train'].copy()
corr_full = train_df[FEATURES + ['log_price']].corr()['log_price'].drop('log_price')
corr_sorted = corr_full.abs().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_feats = corr_sorted.head(25).index
colors = ['#2196F3' if corr_full[f] > 0 else '#F44336' for f in top_feats]
ax.barh(top_feats, corr_full[top_feats], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 25 features by correlation with log_price', fontsize=12)
ax.set_xlabel('Pearson correlation')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap of top numeric features
top_num_features = corr_sorted.head(12).index.tolist()
fig, ax = plt.subplots(figsize=(11, 9))
corr_matrix = train_df[top_num_features + ['log_price']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Correlation heatmap — top features + target')
plt.tight_layout()
plt.savefig('fig_corr_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Price distribution by number of amenities
train_df['amenities_bin'] = pd.cut(train_df['amenities_count'],
                                    bins=[0,5,10,15,20,30,90],
                                    labels=['1-5','6-10','11-15','16-20','21-30','30+'])
fig, ax = plt.subplots(figsize=(10, 4))
amenities_price = train_df.groupby('amenities_bin', observed=True)['log_price'].mean()
ax.bar(amenities_price.index.astype(str), amenities_price.values, color='#7986CB')
ax.set_title('Mean log_price by amenities count bucket')
ax.set_xlabel('Number of amenities')
ax.set_ylabel('Mean log_price')
plt.tight_layout()
plt.savefig('fig_amenities_price.png', dpi=120, bbox_inches='tight')
plt.show()
print("More amenities → higher price, but diminishing returns above ~20.")

## 6. Baseline models

Before anything fancy, establish baselines. RMSE on log_price is the metric — in practical terms, RMSE of 0.4 means predictions are off by roughly e^0.4 ≈ 50% of the true price on average.

In [ ]:
# Prepare training matrix
TARGET = 'log_price'

X = train_df[FEATURES].copy()
y = train_df[TARGET].copy()

imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(model, X, y, cv):
    scores = cross_val_score(model, X, y, cv=cv,
                             scoring='neg_root_mean_squared_error')
    return -scores.mean(), scores.std()

# Baseline 1: predict global mean
baseline_pred = np.full(len(y), y.mean())
print(f"Predict mean baseline RMSE:  {np.sqrt(mean_squared_error(y, baseline_pred)):.4f}")

# Baseline 2: Ridge on all features
ridge = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))])
rmse, std = rmse_cv(ridge, X_imp, y, cv)
print(f"Ridge regression RMSE:        {rmse:.4f} ± {std:.4f}")

## 7. Model comparison

In [ ]:
results = {}

# Ridge
ridge = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=10.0))])
rmse, std = rmse_cv(ridge, X_imp, y, cv)
results['Ridge'] = {'rmse': rmse, 'std': std}
print(f"Ridge:             RMSE = {rmse:.4f} ± {std:.4f}")

# Lasso (also good for feature selection)
lasso = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.001, max_iter=5000))])
rmse, std = rmse_cv(lasso, X_imp, y, cv)
results['Lasso'] = {'rmse': rmse, 'std': std}
print(f"Lasso:             RMSE = {rmse:.4f} ± {std:.4f}")

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=3,
                            max_features=0.5, random_state=42, n_jobs=-1)
rmse, std = rmse_cv(rf, X_imp, y, cv)
results['RandomForest'] = {'rmse': rmse, 'std': std}
print(f"Random Forest:     RMSE = {rmse:.4f} ± {std:.4f}")

In [ ]:
# Gradient Boosting
gbc = GradientBoostingRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                                 subsample=0.8, min_samples_leaf=5, random_state=42)
rmse, std = rmse_cv(gbc, X_imp, y, cv)
results['GradientBoosting'] = {'rmse': rmse, 'std': std}
print(f"Gradient Boosting: RMSE = {rmse:.4f} ± {std:.4f}")

In [ ]:
# XGBoost / LightGBM
if BOOSTING:
    xgb_model = xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                                   subsample=0.8, colsample_bytree=0.7,
                                   min_child_weight=5, random_state=42)
    rmse, std = rmse_cv(xgb_model, X_imp, y, cv)
    results['XGBoost'] = {'rmse': rmse, 'std': std}
    print(f"XGBoost:           RMSE = {rmse:.4f} ± {std:.4f}")

    lgb_model = lgb.LGBMRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.7,
                                    min_child_samples=10, random_state=42, verbose=-1)
    rmse, std = rmse_cv(lgb_model, X_imp, y, cv)
    results['LightGBM'] = {'rmse': rmse, 'std': std}
    print(f"LightGBM:          RMSE = {rmse:.4f} ± {std:.4f}")
else:
    print("XGBoost/LightGBM not available. Install with: pip install xgboost lightgbm")

In [ ]:
# Comparison plot
results_df = pd.DataFrame(results).T.sort_values('rmse')
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2196F3' if i == 0 else '#90CAF9' for i in range(len(results_df))]
bars = ax.barh(results_df.index, results_df['rmse'], color=colors, xerr=results_df['std'], capsize=4)
ax.set_title('5-fold CV RMSE by model (lower is better)', fontsize=12)
ax.set_xlabel('RMSE (log_price)')
ax.axvline(results_df['rmse'].max(), color='gray', linestyle=':', linewidth=0.8)
for i, (idx, row) in enumerate(results_df.iterrows()):
    ax.text(row['rmse'] + 0.003, i, f"{row['rmse']:.4f}", va='center', fontsize=9)
plt.tight_layout()
plt.savefig('fig_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Things that didn't help as expected

Worth documenting what was tried but didn't move the needle.

In [ ]:
# Experiment 1: Does description sentiment actually help?
# The luxury_score feature tries to capture premium listing language.
# Let's check if it adds anything over the base model.

X_with = X_imp.copy()
X_without = X_imp.drop(columns=['desc_luxury_score', 'desc_length', 'desc_word_count',
                                  'name_length'], errors='ignore')

rf_quick = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

rmse_with, _ = rmse_cv(rf_quick, X_with, y, cv)
rmse_without, _ = rmse_cv(rf_quick, X_without, y, cv)
print(f"RF with text features:    RMSE = {rmse_with:.4f}")
print(f"RF without text features: RMSE = {rmse_without:.4f}")
print()
print("Result: text features add very little. Description is truncated at 1000 chars in this dataset")
print("and 'luxury_score' is too coarse. For a real project you'd want proper NLP (TF-IDF + SVD).")

In [ ]:
# Experiment 2: raw neighbourhood one-hot vs target encoding
# With 1000+ unique neighbourhoods, OHE creates noise. Let's verify target encoding is better.

# Target encoding already in features. Let's try frequency only.
X_freq_only = X_imp.copy()
X_freq_only['neighbourhood_mean_price'] = X_imp['neighbourhood_freq']  # replace with freq

rf_quick2 = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rmse_target, _ = rmse_cv(rf_quick2, X_imp, y, cv)
rmse_freq, _ = rmse_cv(rf_quick2, X_freq_only, y, cv)
print(f"Target encoding neighbourhood: RMSE = {rmse_target:.4f}")
print(f"Frequency encoding only:       RMSE = {rmse_freq:.4f}")
print()
print("Target encoding wins. Frequency is a noisy proxy — knowing a neighbourhood is popular")
print("doesn't tell the model whether it's expensive (SoHo) or not (Brownsville).")

## 9. Hyperparameter tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Tune the best tree model
param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [5, 7, 10],
    'min_samples_leaf': [2, 5, 10],
    'max_features': [0.4, 0.6, 0.8],
    'subsample': [0.7, 0.85, 1.0],
    'learning_rate': [0.03, 0.05, 0.1],
}

gbc_tune = GradientBoostingRegressor(random_state=42)
search = RandomizedSearchCV(gbc_tune, param_grid, n_iter=20, cv=cv,
                             scoring='neg_root_mean_squared_error',
                             random_state=42, n_jobs=-1, verbose=1)
search.fit(X_imp, y)

print(f"\nBest params: {search.best_params_}")
print(f"Best RMSE:   {-search.best_score_:.4f}")

In [ ]:
# Check improvement
gbc_best = search.best_estimator_
rmse_best, std_best = rmse_cv(gbc_best, X_imp, y, cv)
print(f"Tuned GBM RMSE:   {rmse_best:.4f} ± {std_best:.4f}")
print(f"Default GBM RMSE: {results['GradientBoosting']['rmse']:.4f}")

## 10. Error analysis

In [ ]:
from sklearn.model_selection import cross_val_predict

best_model = search.best_estimator_
y_cv_pred = cross_val_predict(best_model, X_imp, y, cv=cv)
residuals = y - y_cv_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Predicted vs actual
axes[0].scatter(y_cv_pred, y, alpha=0.15, s=5, color='#42A5F5')
mn, mx = y.min(), y.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5)
axes[0].set_xlabel('Predicted log_price')
axes[0].set_ylabel('Actual log_price')
axes[0].set_title('Predicted vs Actual (CV)')

# Residuals vs predicted
axes[1].scatter(y_cv_pred, residuals, alpha=0.15, s=5, color='#66BB6A')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted log_price')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Fitted')

# Residual distribution
axes[2].hist(residuals, bins=80, color='#AB47BC', edgecolor='white', linewidth=0.2)
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Count')
axes[2].set_title(f'Residual distribution (RMSE={residuals.std():.3f})')
axes[2].axvline(0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig('fig_residuals.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Where does the model struggle most?
train_diag = train_df.copy()
train_diag['predicted'] = y_cv_pred
train_diag['residual'] = residuals
train_diag['abs_error'] = np.abs(residuals)

print("Mean absolute error by room type:")
print(train_diag.groupby('room_type')['abs_error'].mean().round(3).sort_values(ascending=False).to_string())
print()
print("Mean absolute error by city:")
print(train_diag.groupby('city')['abs_error'].mean().round(3).sort_values(ascending=False).to_string())
print()
print("Mean absolute error by property type (top 6):")
top6 = train_diag['property_type'].value_counts().head(6).index
print(train_diag[train_diag['property_type'].isin(top6)].groupby('property_type')['abs_error'].mean().round(3).sort_values(ascending=False).to_string())

In [ ]:
# Visualize error by price range
train_diag['price_bin'] = pd.cut(train_diag['log_price'],
                                  bins=[2, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 8],
                                  labels=['<$33','$33-55','$55-90','$90-148','$148-245','$245-403','>$403'])

fig, ax = plt.subplots(figsize=(10, 4))
error_by_bin = train_diag.groupby('price_bin', observed=True)['abs_error'].mean()
ax.bar(error_by_bin.index.astype(str), error_by_bin.values, color='#EF5350')
ax.set_title('Mean absolute error by price range')
ax.set_xlabel('Price range')
ax.set_ylabel('Mean |error| in log_price')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('fig_error_by_price.png', dpi=120, bbox_inches='tight')
plt.show()
print("Models typically underestimate very high-end listings — makes sense, luxury is hard to capture from text alone.")

## 11. Feature importance

In [ ]:
# Train RF on full data for importance
rf_full = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=3,
                                 max_features=0.5, random_state=42, n_jobs=-1)
rf_full.fit(X_imp, y)

feat_imp = pd.Series(rf_full.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.head(20).plot(kind='barh', ax=ax, color='#5C6BC0')
ax.set_title('Random Forest feature importance (top 20)', fontsize=12)
ax.set_xlabel('Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top 10 features:")
print(feat_imp.head(10).round(4).to_string())

In [ ]:
# Partial dependence: how does log_price change with accommodates?
accom_range = np.arange(1, 13)
X_template = X_imp.copy()
X_template_mean = X_template.median()

pdp_preds = []
for acc in accom_range:
    X_mod = pd.DataFrame([X_template_mean] * 200)
    X_mod['accommodates'] = acc
    pdp_preds.append(rf_full.predict(X_mod).mean())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(accom_range, pdp_preds, marker='o', linewidth=2, color='#26A69A')
ax.set_title('Partial dependence: accommodates → log_price')
ax.set_xlabel('Accommodates')
ax.set_ylabel('Predicted log_price (avg over dataset)')
ax.set_xticks(accom_range)
plt.tight_layout()
plt.savefig('fig_pdp_accommodates.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Final predictions

Training the best model on the full training set, then predicting on test.

In [ ]:
# Choose best model based on CV
# If XGBoost/LightGBM available, pick best; otherwise use GBM
print("CV results summary:")
for name, res in sorted(results.items(), key=lambda x: x[1]['rmse']):
    print(f"  {name:20s}: RMSE = {res['rmse']:.4f} ± {res['std']:.4f}")

# Select the winner
best_model_name = min(results, key=lambda k: results[k]['rmse'])
print(f"\nBest model: {best_model_name}")

In [ ]:
# Prepare test features
test_df = df[df['_split'] == 'test'].copy()
X_test = test_df[FEATURES].copy()
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=FEATURES)
print(f"Test feature matrix: {X_test_imp.shape}")
print(f"Missing after imputation: {X_test_imp.isnull().sum().sum()}")

In [ ]:
# Train final model on all training data
if BOOSTING and best_model_name == 'LightGBM':
    final_model = lgb.LGBMRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                                     subsample=0.8, colsample_bytree=0.7,
                                     min_child_samples=10, random_state=42, verbose=-1)
elif BOOSTING and best_model_name == 'XGBoost':
    final_model = xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.7,
                                    min_child_weight=5, random_state=42)
else:
    # GBM tuned
    final_model = search.best_estimator_

final_model.fit(X_imp, y)
y_pred = final_model.predict(X_test_imp)

print(f"Predictions range: {y_pred.min():.3f} — {y_pred.max():.3f}")
print(f"Predictions mean:  {y_pred.mean():.3f} (train mean: {y.mean():.3f})")
print(f"Predictions std:   {y_pred.std():.3f} (train std: {y.std():.3f})")

In [ ]:
# Sanity check: predicted log_price distribution should match train
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(y, bins=60, alpha=0.5, color='#42A5F5', label='Train (actual)', density=True)
ax.hist(y_pred, bins=60, alpha=0.5, color='#EF5350', label='Test (predicted)', density=True)
ax.set_xlabel('log_price')
ax.set_ylabel('Density')
ax.set_title('Train actual vs test predicted distribution')
ax.legend()
plt.tight_layout()
plt.savefig('fig_pred_dist.png', dpi=120, bbox_inches='tight')
plt.show()

## 13. Submission file

In [ ]:
# Match sample format: columns are 'Unnamed: 0' and 'logpred'
print("Expected format:")
print(sample.head(5).to_string())
print()

submission = pd.DataFrame({
    'Unnamed: 0': test_df['id'].values,
    'logpred': y_pred
})

print("Our output:")
print(submission.head(5).to_string())
print()
print(f"Row count: {len(submission)} (expected: {len(sample)})")
assert len(submission) == len(sample), f"Row count mismatch! {len(submission)} vs {len(sample)}"

# Verify the ids match
sample_ids = set(sample['Unnamed: 0'].values)
our_ids    = set(submission['Unnamed: 0'].values)
print(f"IDs matching sample: {len(sample_ids & our_ids)} / {len(sample_ids)}")

In [ ]:
# Sort to match sample order
submission = submission.set_index('Unnamed: 0').reindex(sample['Unnamed: 0']).reset_index()
submission.columns = ['Unnamed: 0', 'logpred']

submission.to_csv('predictions_airbnb.csv', index=False)
print("Saved: predictions_airbnb.csv")
print(submission['logpred'].describe().round(3).to_string())

## 14. Conclusion and limitations

The room type, accommodates, and city are by far the strongest predictors — they explain the structural price variation. Amenities add a moderate signal, especially elevator, doorman, and pool which distinguish luxury listings. Geographic features (lat/lon, zipcode mean price, neighbourhood target encoding) matter significantly within cities.

The model struggles most at the high end (luxury Villas, high-end SF/DC entire apartments) — not surprising, since these rely on unquantifiable factors: interior design, reputation, one-of-a-kind views. The description is truncated at 1000 characters here which limits any NLP-based premium signal extraction.

**What didn't work:**
- Keyword sentiment scoring on descriptions (too noisy at 1000 char limit)
- Frequency-only neighbourhood encoding (target encoding clearly better)
- Including zipcode as raw string (messy, needed 5-digit cleaning first)

**What would help with more data or time:**
- TF-IDF on full description text + truncated SVD embeddings (proper NLP)
- Listing photo count or quality (not available here, but huge in real Airbnb pricing)
- Seasonal pricing variation (no date of listing snapshot available)
- Review text sentiment (not included in this dataset)
- A stacking ensemble: Ridge predictions as a feature in the tree model

Realistic RMSE target on this task: ~0.38–0.44. Getting below 0.38 likely requires NLP or external data.
